# Task 1 Kaggle Training Notebook

Notebook nay dung de train `Gemma 4 26B A4B` cho bai toan Task 1 tren `AvaMERG + ESConv`.

Thu tu chay khuyen nghi:
1. Install dependencies
2. Login Hugging Face
3. Download data
4. Inspect samples
5. Dump prompts
6. Smoke test 1 step
7. Train that su


In [ ]:
import os
REPO_URL = "https://github.com/QuangVoAI/multimodal-empathy-mental-health.git"
REPO_DIR = "/kaggle/working/multimodal-empathy-mental-health"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd /kaggle/working/multimodal-empathy-mental-health
%pip install --upgrade pip setuptools wheel
%pip install --no-cache-dir --force-reinstall -r requirements.txt


## Important: restart the Kaggle kernel now

Sau khi cell cai package chay xong, hay **Restart Session / Restart Kernel** roi moi chay tiep tu cell dang nhap Hugging Face.
Buoc nay tranh loi import lech version giua package cu va moi.


In [ ]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN"
login(HF_TOKEN)


In [ ]:
from transformers import AutoProcessor

MODEL_ID = "google/gemma-4-26B-A4B-it"
processor = AutoProcessor.from_pretrained(MODEL_ID, token=True)
tokenizer = getattr(processor, "tokenizer", None)
print("Processor loaded:", MODEL_ID)
if tokenizer is not None:
    print("Pad token:", tokenizer.pad_token)
    print("EOS token:", tokenizer.eos_token)


In [ ]:
!pip install -U huggingface_hub
!bash scripts/download_avamerg.sh
!bash scripts/download_esconv.sh
!mkdir -p outputs/sft


In [ ]:
import json
from pathlib import Path

ava = json.loads(Path("data/raw/avamerg/train.json").read_text(encoding="utf-8"))
esc = json.loads(Path("data/raw/esconv/ESConv.json").read_text(encoding="utf-8"))
print("AvaMERG samples:", len(ava))
print("AvaMERG first keys:", list(ava[0].keys()))
print("ESConv dialogues:", len(esc))
print("ESConv first keys:", list(esc[0].keys()))


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training

base_cfg = dict(
    model_name_or_path=MODEL_ID,
    avamerg_root="data/raw/avamerg",
    avamerg_split="train",
    avamerg_text_only=True,
    esconv_json="data/raw/esconv/ESConv.json",
    output_dir="outputs/sft/debug_joint",
    max_length=2048,
    max_response_tokens=256,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=1.0,
    logging_steps=1,
    save_steps=20,
    warmup_ratio=0.03,
    max_steps=-1,
    use_lora=True,
    load_in_4bit=True,
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    report_to="none",
    dump_example_prompts=True,
)

dump_args = Namespace(**base_cfg)
run_training(dump_args)


In [ ]:
!sed -n '1,200p' outputs/sft/debug_joint/example_prompts.json


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training

smoke_cfg = dict(base_cfg)
smoke_cfg.update({
    "output_dir": "outputs/sft/joint_smoke",
    "dump_example_prompts": False,
    "max_steps": 1,
})

smoke_args = Namespace(**smoke_cfg)
run_training(smoke_args)


In [ ]:
from argparse import Namespace
from scripts.train_sft import run_training

# Uncomment and adjust after the smoke run is stable.
# train_cfg = dict(base_cfg)
# train_cfg.update({
#     "output_dir": "outputs/sft/joint_lora",
#     "dump_example_prompts": False,
#     "max_steps": -1,
#     "gradient_accumulation_steps": 8,
#     "logging_steps": 10,
#     "save_steps": 100,
# })
# train_args = Namespace(**train_cfg)
# run_training(train_args)
